In [2]:
from pathlib import Path

import nltk
import jiwer
import pandas as pd

In [3]:
PREDICTION_DIR = Path("/Users/eduardo/argmax/Dolphin/results-fox-page-en/recognition_json")
GT_FILE = Path("/Users/eduardo/argmax/Dolphin/focus_benchmark_test/en_page_ocr.json")
EDIT_DISTANCES_FILE = Path("/Users/eduardo/argmax/Dolphin/results-fox-page-en/edit_distances.csv")

In [4]:
def load_ground_truth(file_path: Path) -> pd.DataFrame:
    df = pd.read_json(file_path)
    df["conversations"] = df["conversations"].apply(lambda x: x[1]["value"])
    df['image'] = df['image'].apply(lambda x: Path(x).stem)
    df = df.rename(columns={"conversations": "text"})
    return df

def load_prediction_sample(file_path: Path) -> str:
    df = pd.read_json(file_path)
    return "\n".join(df["text"].tolist())

def load_predictions(dir_path: Path) -> pd.DataFrame:
    predictions = []
    for file_path in dir_path.glob("*.json"):
        predictions.append((file_path.stem, load_prediction_sample(file_path)))
    return pd.DataFrame(predictions, columns=["image", "text"])

def compute_edit_distance(pred: str, gt: str) -> float:
    return nltk.edit_distance(pred, gt) / max(len(pred), len(gt))

In [6]:
gt_df = load_ground_truth(GT_FILE)
predictions_df = load_predictions(PREDICTION_DIR)
ed_df = pd.read_csv(EDIT_DISTANCES_FILE)

df = gt_df.merge(predictions_df, on="image", how="left", suffixes=("_gt", "_pred")).dropna().drop(columns=["len"])
cer = jiwer.cer(reference=df["text_gt"].tolist(), hypothesis=df["text_pred"].tolist())
print(f"CER: {cer:.2%}")

CER: 5.57%


In [13]:
from transformers import VisionEncoderDecoderModel, AutoProcessor

In [14]:
model = VisionEncoderDecoderModel.from_pretrained("ByteDance/Dolphin")
processor = AutoProcessor.from_pretrained("ByteDance/Dolphin")

In [12]:
model.encoder.config

DonutSwinConfig {
  "_attn_implementation_autoset": true,
  "attention_probs_dropout_prob": 0.0,
  "depths": [
    2,
    2,
    14,
    2
  ],
  "drop_path_rate": 0.1,
  "embed_dim": 128,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 1024,
  "image_size": [
    896,
    896
  ],
  "initializer_range": 0.02,
  "layer_norm_eps": 1e-05,
  "mlp_ratio": 4.0,
  "model_type": "donut-swin",
  "num_channels": 3,
  "num_heads": [
    4,
    8,
    16,
    32
  ],
  "num_layers": 4,
  "patch_size": 4,
  "qkv_bias": true,
  "transformers_version": "4.47.0",
  "use_absolute_embeddings": false,
  "window_size": 7
}